# 00 视频 → Landmarks 提取

从 good rep 视频中提取 MediaPipe **world landmarks**（33 点 × 3 坐标，米制），保存为 `.npy` 供 `01_data_prep.ipynb` 使用。

**流程**：
1. 挂载 Google Drive
2. 读取 `good_reps_videos/` 下的视频
3. 逐帧用 MediaPipe Pose Landmarker 检测
4. 保存 world landmarks 到 `good_reps/*.npy`，shape `[T, 33, 3]`

In [ ]:
# 安装依赖（Colab 首次运行）
!pip install -q mediapipe opencv-python

In [ ]:
# 环境与路径
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/ai_form_1")  # Colab clone 路径
DATA_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks_AIForm/Model20260305/ai_form_coach_data")
if not Path("/content/drive").exists():
  DATA_ROOT = PROJECT_ROOT / "tools" / "train_denoiser" / "data"

VIDEOS_DIR = DATA_ROOT / "good_reps_videos"
OUTPUT_DIR = DATA_ROOT / "good_reps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"视频目录: {VIDEOS_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

In [ ]:
# Colab: 挂载 Google Drive
try:
  from google.colab import drive
  drive.mount("/content/drive")
except ImportError:
  print("非 Colab 环境，跳过 Drive 挂载")

In [ ]:
# 下载 MediaPipe Pose Landmarker 模型（若本地无）
DATA_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_PATH = DATA_ROOT / "pose_landmarker.task"
if not MODEL_PATH.exists():
  import urllib.request
  url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
  urllib.request.urlretrieve(url, str(MODEL_PATH))
  print(f"已下载模型: {MODEL_PATH}")
else:
  print(f"使用已有模型: {MODEL_PATH}")

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def extract_landmarks_from_video(video_path: Path, model_path: Path) -> np.ndarray:
  """
  从视频提取 world landmarks，返回 [T, 33, 3] float32。
  """
  cap = cv2.VideoCapture(str(video_path))
  fps = cap.get(cv2.CAP_PROP_FPS) or 30
  frame_ms = 1000.0 / fps

  base_options = python.BaseOptions(model_asset_path=str(model_path))
  options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
  )
  landmarker = vision.PoseLandmarker.create_from_options(options)

  frames_data = []
  frame_idx = 0

  while True:
    ret, frame = cap.read()
    if not ret:
      break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    timestamp_ms = int(frame_idx * frame_ms)

    result = landmarker.detect_for_video(mp_image, timestamp_ms)

    if result.pose_world_landmarks:
      lm = result.pose_world_landmarks[0]
      arr = np.array([[p.x, p.y, p.z] for p in lm], dtype=np.float32)
      frames_data.append(arr)
    else:
      # 未检测到姿态时用零填充
      frames_data.append(np.zeros((33, 3), dtype=np.float32))

    frame_idx += 1

  cap.release()
  landmarker.close()

  if not frames_data:
    return np.zeros((0, 33, 3), dtype=np.float32)
  return np.stack(frames_data, axis=0)

In [ ]:
# 遍历视频并提取
video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
video_paths = [p for p in VIDEOS_DIR.iterdir() if p.suffix.lower() in video_exts] if VIDEOS_DIR.exists() else []

if not video_paths:
  print(f"未找到视频，请将视频放入: {VIDEOS_DIR}")
  print("支持格式: mp4, avi, mov, mkv, webm")
else:
  for vp in video_paths:
    out_path = OUTPUT_DIR / (vp.stem + ".npy")
    print(f"处理: {vp.name} -> {out_path.name}", end=" ")
    arr = extract_landmarks_from_video(vp, MODEL_PATH)
    np.save(out_path, arr)
    print(f"| {arr.shape[0]} 帧")
  print(f"\n完成，共 {len(video_paths)} 个 rep 已保存到 {OUTPUT_DIR}")

In [ ]:
# 校验：加载一个 .npy 查看 shape
npy_files = list(OUTPUT_DIR.glob("*.npy"))
if npy_files:
  sample = np.load(npy_files[0])
  print(f"样本: {npy_files[0].name}")
  print(f"shape: {sample.shape} (应为 [T, 33, 3])")
else:
  print("暂无 .npy 文件，请先运行上方 cell 提取视频")